# PD correction examples at acquired focal positions

Rows show all four focal positions from the acquisition, with zero phase diversity. Columns show the uncorrected state and states after loops 1–3. Images are regenerated with fresh Poisson noise at the saved fixed exposure; they are not exact original noise realizations. The final column is simulated: no acquisition occurred after loop 3.

Four focal planes are computed together for each state (four forward calls per example), without rerunning optimization or simulating diversity stacks. Intensity is normalized to the brightest pixel across the whole grid; plasma phase insets are fixed at ±2π radians.

Slide sizing: both image grids and RMS graphs are 6 × 6 inches, for side-by-side placement on a widescreen slide. PNGs use 300 DPI; PDFs retain vector text.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not ((ROOT / "utils").is_dir() and (ROOT / "optimization").is_dir()):
    if ROOT == ROOT.parent:
        raise RuntimeError("Run this notebook inside the repository.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
EXPERIMENT_DIR = ROOT / "systematic_testing" / "correction_examples"
FIGURES_DIR = EXPERIMENT_DIR / "figures"
RESULTS_DIR = EXPERIMENT_DIR / "results"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
import importlib
from systematic_testing.correction_examples import plot_pd_correction_examples as correction_examples
importlib.reload(correction_examples)  # Refresh styles and fixed phase limits in an existing kernel.
from systematic_testing.correction_examples.plot_pd_correction_examples import DEFAULT_DATA, DEFAULT_CASES, plot_correction_example
RESULTS_PATH = DEFAULT_DATA
EXAMPLE_CASES = DEFAULT_CASES + [
    (0.15, 5., 0),
    (0.15, 10., 0),
    (0.15, 15., 0),
    (0.30, 10., 2),
]  # (initial RMS, calibration SNR, zero-based trial)

# Match the sweep notebooks.
plt.style.use("bmh")
plt.rcParams.update({"figure.figsize": [548 / 72, 390 / 72],
                     "font.size": 12, "text.usetex": True, "figure.dpi": 300})


## Inspect acquisition metadata

This only reads the archive; it does not simulate images.


In [ ]:
with np.load(RESULTS_PATH, allow_pickle=False) as saved:
    print("Acquired focal positions [µm]:", saved["focal_positions"] * 1000)
    print("Saved correction loops:", saved["ao_loops"])
    print("Image arrays stored:", [key for key in saved.files if saved[key].ndim >= 5 and key != "slm_strengths"])


## Generate correction examples
Edit `EXAMPLE_CASES` above to choose RMS, SNR, and zero-based trial. The seeded fresh noise is repeatable.

In [ ]:
OUTPUT_DIR = ROOT / "systematic_testing" / "correction_examples" / "figures" / "pd_correction_examples_acquired_planes"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with np.load(RESULTS_PATH, allow_pickle=False) as saved:
    for rms, snr, trial in EXAMPLE_CASES:
        exposure = float(saved["exposures"][
            np.flatnonzero(np.isclose(saved["initial_rms"], rms))[0],
            np.flatnonzero(np.isclose(saved["target_snr"], snr))[0], trial])
        print(f"RMS {rms:g}, SNR {snr:g}, trial {trial+1}: exposure {exposure:.6g}")
        fig, axes = plot_correction_example(saved, rms, snr, trial, noisy=True, acquired_planes=True)
        stem = f"rms{rms:g}_snr{snr:g}_trial{trial+1}_noisy_acquired_planes"
        for extension in ("png", "pdf"):
            fig.savefig(OUTPUT_DIR / f"{stem}.{extension}", dpi=300)
        plt.show()
        plt.close(fig)


## Generated previews

![Example](figures/pd_correction_examples_acquired_planes/rms0.15_snr100_trial1_noisy_acquired_planes.png)

![Example](figures/pd_correction_examples_acquired_planes/rms0.15_snr100_trial2_noisy_acquired_planes.png)

![Example](figures/pd_correction_examples_acquired_planes/rms0.3_snr5_trial3_noisy_acquired_planes.png)

![Example](figures/pd_correction_examples_acquired_planes/rms0.3_snr100_trial3_noisy_acquired_planes.png)


### Additional lower-SNR examples

![RMS 0.15, SNR 5, trial 1](figures/pd_correction_examples_acquired_planes/rms0.15_snr5_trial1_noisy_acquired_planes.png)

![RMS 0.15, SNR 10, trial 1](figures/pd_correction_examples_acquired_planes/rms0.15_snr10_trial1_noisy_acquired_planes.png)

![RMS 0.15, SNR 15, trial 1](figures/pd_correction_examples_acquired_planes/rms0.15_snr15_trial1_noisy_acquired_planes.png)

![RMS 0.3, SNR 10, trial 3](figures/pd_correction_examples_acquired_planes/rms0.3_snr10_trial3_noisy_acquired_planes.png)



Inset phase limits are fixed at $-2\pi$ and $2\pi$ radians. Rerun the setup/import cell to reload the plotting helper in an existing kernel.


## Correction RMS versus optimizer iterations

Curves use the recovered per-iteration coefficient estimates from `pd_rms_snr_sweep_random100_iteration_histories.npz`. For each loop, RMS is calculated between the current residual truth and the optimizer estimate. It is the residual that subtracting the full estimate would produce, before clipping. Black crosses show actual applied residual RMS after the loop correction, including clipping.

Dashed lines mark loop starts/ends. Each loop starts with its own initializer; curves are separate at shared boundaries. The iteration axis counts the recorded optimization iterations, excluding random-start screening evaluations. No optimization is rerun.


In [ ]:
from systematic_testing.correction_examples import plot_pd_iteration_rms as iteration_plots
importlib.reload(iteration_plots)
with np.load(RESULTS_PATH, allow_pickle=False) as saved:
    for rms, snr, trial in EXAMPLE_CASES:
        fig, ax = iteration_plots.plot_iteration_rms(saved, rms, snr, trial)
        stem = f"rms{rms:g}_snr{snr:g}_trial{trial+1}_iteration_rms"
        for extension in ("png", "pdf"):
            fig.savefig(OUTPUT_DIR / f"{stem}.{extension}", dpi=300)
        plt.show()
        plt.close(fig)


## Saved per-iteration RMS plots

![Per-iteration RMS error](figures/pd_correction_examples_acquired_planes/rms0.15_snr100_trial1_iteration_rms.png)

![Per-iteration RMS error](figures/pd_correction_examples_acquired_planes/rms0.15_snr100_trial2_iteration_rms.png)

![Per-iteration RMS error](figures/pd_correction_examples_acquired_planes/rms0.3_snr5_trial3_iteration_rms.png)

![Per-iteration RMS error](figures/pd_correction_examples_acquired_planes/rms0.3_snr100_trial3_iteration_rms.png)

![Per-iteration RMS error](figures/pd_correction_examples_acquired_planes/rms0.15_snr5_trial1_iteration_rms.png)

![Per-iteration RMS error](figures/pd_correction_examples_acquired_planes/rms0.15_snr10_trial1_iteration_rms.png)

![Per-iteration RMS error](figures/pd_correction_examples_acquired_planes/rms0.15_snr15_trial1_iteration_rms.png)

![Per-iteration RMS error](figures/pd_correction_examples_acquired_planes/rms0.3_snr10_trial3_iteration_rms.png)

